
## cấu trúc
```
Controller
   ↓
Shared Method (HandleCommand)
   ↓
Step methods
   ├─ InitTracking()
   ├─ Validate()
   ├─ EnqueueJob()
   └─ HandleError()
   ↓
Response
```

## flow
```
http request
   ↓
controller endpoint
   ↓
shared-method (handle)
   ↓
validate
   ↓
business logic
   ↓
enqueue job / response
```

## cấu trúc thư mục (tham khảo)
```
discounts
│
├─ controllers
│  └─ skytransfer
│      └─ SkyTransferDiscountController.cs
│
├─ application
│  └─ DiscountApplicationService.cs
│
├─ abstractions
│  └─ SkyCrmDiscountCommandBase.cs
│
├─ models
│  ├─ commands
│  │   ├─ SkyCrmDiscountCreateCommand.cs
│  │   ├─ SkyCrmDiscountUpdateCommand.cs
│  │   └─ SkyCrmDiscountDeleteCommand.cs
│  │
│  └─ responses
│      └─ SkyCrmDiscountIntegrationCommandResponse.cs
│
├─ steps
│  ├─ tracking
│  │   ├─ InitTrackingStep.cs
│  │   └─ FinishTrackingStep.cs
│  │
│  ├─ validation
│  │   └─ DiscountValidationService.cs
│  │
│  └─ background
│      └─ EnqueueDiscountJobStep.cs
│
├─ background
│  └─ jobs
│      └─ SkyCrmDiscountJob.cs
│
└─ integrations
   └─ skytransfer
       └─ SkyTransferDiscountIntegrationService.cs
```

In [1]:
using System;
using System.Threading.Tasks;

// =========================
// base request
// =========================

public abstract class DiscountRequestBase
{
    public string Name { get; set; }
}

public class DiscountCreateCommand : DiscountRequestBase { }

public class DiscountUpdateCommand : DiscountRequestBase { }

public class DiscountDeleteCommand : DiscountRequestBase { }


// =========================
// response
// =========================

public class CommandResponse
{
    public bool Success { get; set; }
    public string Message { get; set; }

    public static CommandResponse Ok()
        => new CommandResponse { Success = true };

    public static CommandResponse Fail(string msg)
        => new CommandResponse { Success = false, Message = msg };
}


// =========================
// service
// =========================

public class DiscountService
{

    public async Task<CommandResponse> HandleCommand<T>(T command)
        where T : DiscountRequestBase
    {
        try
        {
            var validate = Validate(command);

            if (!validate.Success)
                return validate;

            await EnqueueJob(command);

            return CommandResponse.Ok();
        }
        catch (Exception ex)
        {
            return CommandResponse.Fail(ex.Message);
        }
    }


    private CommandResponse Validate(DiscountRequestBase command)
    {
        if (string.IsNullOrWhiteSpace(command.Name))
            return CommandResponse.Fail("name required");

        return CommandResponse.Ok();
    }


    private Task EnqueueJob(DiscountRequestBase command)
    {
        Console.WriteLine($"enqueue job for {command.Name}");
        return Task.CompletedTask;
    }

}


// =========================
// thin controller
// =========================

public class DiscountController
{
    private readonly DiscountService _service = new();


    public async Task Create(DiscountCreateCommand command)
    {
        var res = await _service.HandleCommand(command);
        Console.WriteLine(res.Success);
    }


    public async Task Update(DiscountUpdateCommand command)
    {
        var res = await _service.HandleCommand(command);
        Console.WriteLine(res.Success);
    }


    public async Task Delete(DiscountDeleteCommand command)
    {
        var res = await _service.HandleCommand(command);
        Console.WriteLine(res.Success);
    }

}


// =========================
// run demo
// =========================

var controller = new DiscountController();

await controller.Create(new DiscountCreateCommand
{
    Name = "summer promo"
});

enqueue job for summer promo
True
